> 在线讲义：[第 19 章 · 巨大激活与 attention sink：为什么大家都盯着第一个词](https://wikieden.github.io/beyond-transformer/lessons/19-attention-sink.html) · 本书《超越 Transformer》

# 第 19 章 · 巨大激活与 attention sink：为什么大家都盯着第一个词

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/19-attention-sink.html) · 仓库里 `beyond-transformer/notes/lessons/19-attention-sink.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 巨大激活 | 最大 / 中位数 | `massive()` | 相差 1,322 倍 |
| 第二步 sink | 句首的权重 | `compare()["sink 拿到的权重"]` | 0.9447 |
| 第三步 滑窗 vs StreamingLLM | 输出差 | `compare()` | 0.3857 vs 0.0153 |

停在第一个 token 上（它的 value 很小，等于什么都不加）；滑窗推理把第一个 token 踢出去，权重被迫摊到别的 token 上，
输出就变了；StreamingLLM 留住开头几个 token 就修好了。
在自己训的小模型里观察 attention sink 是里程碑 T-M2 的一部分。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
N, W, D = 24, 8, 4

**函数 `toy`**：位置 0 是 sink：分数高（巨大激活让它和所有 query 都对得上）、value 接近 0；其余 token 分数小、value 随机。

In [ ]:
def toy(seed=0, n=N, d=D):
    """位置 0 是 sink：分数高（巨大激活让它和所有 query 都对得上）、value 接近 0；其余 token 分数小、value 随机。"""
    rng = np.random.default_rng(seed)
    scores = rng.normal(0, 0.5, n)
    scores[0] = 6.0
    values = rng.normal(0, 1, (n, d))
    values[0] = 0.02 * values[0]
    return scores, values

**函数 `attend`**：小工具。

In [ ]:
def attend(scores, values, keep):
    s = scores[keep]
    w = np.exp(s - s.max())
    w /= w.sum()
    return w, w @ values[keep]

**函数 `compare`**：最后一个位置：全看 / 只看最近 W 个 / 最近 W 个 + 开头 n_sink 个。

In [ ]:
def compare(seed=0, n_sink=4):
    """最后一个位置：全看 / 只看最近 W 个 / 最近 W 个 + 开头 n_sink 个。"""
    s, v = toy(seed)
    full_w, full = attend(s, v, np.arange(N))
    _, win = attend(s, v, np.arange(N - W, N))
    keep = np.r_[np.arange(n_sink), np.arange(N - W, N)]
    _, sink = attend(s, v, keep)
    return {
        "sink 拿到的权重": float(full_w[0]),
        "全看的输出范数": float(np.linalg.norm(full)),
        "滑窗（踢掉开头）与全看的差": float(np.linalg.norm(win - full)),
        f"滑窗 + 开头 {n_sink} 个与全看的差": float(np.linalg.norm(sink - full)),
    }

**函数 `massive`**：巨大激活：某个 token 在少数几维上的值是其余的上千倍。

In [ ]:
def massive(seed=0, d=64):
    """巨大激活：某个 token 在少数几维上的值是其余的上千倍。"""
    rng = np.random.default_rng(seed)
    h = rng.normal(0, 1, (6, d))
    h[0, 7], h[0, 41] = 900.0, -650.0
    med = np.median(np.abs(h))
    return float(np.abs(h).max()), float(med)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 巨大激活")
mx, med = massive()
print(f"  6 个 token × 64 维：最大的 |值| {mx:.0f}，中位数 {med:.3f}，相差 {mx / med:,.0f} 倍（只在第 0 个 token 的两维上）")
print("\n第二步 attention sink 与滑窗")
for k, val in compare().items():
    print(f"  {k}：{val:.4f}")

## 3. 代码里才看得到的

- **sink 的分数和 value 都是手写的**：真实模型里它们是训练出来的，靠的就是第一步那种巨大激活。
- **位置编号要保持一致**：StreamingLLM 实际实现里，cache 里 token 的位置是按“在 cache 里的位置”重新编号的，而不是原文里的绝对位置，否则 RoPE 的相对距离会超出训练范围（第 17 章）；这里的玩具没有位置编码，看不到这一点。

## 4. 自己改一改

- 把 `n_sink` 改成 1 和 8，输出差怎么变？
- 把句首的分数从 6 降到 2，滑窗踢掉句首时还会差那么多吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 两个维度大上万倍，像固定偏置，句首、句号最常见。
> 票必须投完，没人可投就投弃权箱：句首 value 近零，权重停那里就是空操作。
> 滑窗踢掉句首，多余的票摊给真内容，输出全乱；留住开头四个，就稳了。